In [ ]:
%pip install llama_index llama-index-readers-web

In [2]:
from IPython.display import HTML, display
from llama_index.core import VectorStoreIndex, get_response_synthesizer
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.prompts import RichPromptTemplate
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.workflow import (
    Context,
    Event,
    StartEvent,
    StopEvent,
    Workflow,
    step,
)
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI
from llama_index.readers.web import SimpleWebPageReader

from lettucedetect_api.client import LettuceClient
from lettucedetect_api.models import TokenDetectionItem

In [3]:
def display_output(predictions: list[TokenDetectionItem]) -> None:
    text = [item.token for item in predictions]
    colors = [f"rgba(255, 0, 0, {item.hallucination_score * 0.8})" for item in predictions]
    html_elements = [
        f'<span style="background-color: {color};">{text}</span>'
        for color, text in zip(colors, text)
    ]
    html = "".join(html_elements)
    display(HTML(html))


def create_index(url: str) -> VectorStoreIndex:
    documents = SimpleWebPageReader(html_to_text=True).load_data([url])
    splitter = SentenceSplitter(chunk_size=300, chunk_overlap=60)
    embed_model = OpenAIEmbedding(model_name="text-embedding-3-small")
    return VectorStoreIndex.from_documents(
        documents=documents,
        embed_model=embed_model,
        transformations=[splitter],
    )

In [4]:
class AnswerEvent(Event):
    """Result of a RAG query."""

    question: str
    answer: str
    context: list[str]

In [5]:
prompt_template = RichPromptTemplate(
    '{% chat role="system" %}\n'
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer the question. "
    "If you don't know the answer, just say that you don't know. "
    "Always add facts about sea life and related those facts to the context in a funny and creative way. "
    "Don't use emojis."
    "{% endchat %}\n"
    '{% chat role="user" %}\n'
    "Context: {{ context_str }}\n\nQuestion: {{ query_str }}\n"
    "{% endchat %}\n"
)

In [6]:
class RAGWorkflow(Workflow):
    @step
    async def query(self, ctx: Context, ev: StartEvent) -> AnswerEvent:
        question = ev.get("question")
        index = ev.get("index")
        llm = OpenAI(model="gpt-4.1-nano")
        retriever = index.as_retriever(similarity_top_k=2)
        synthesizer = get_response_synthesizer(
            llm=llm, response_mode="simple_summarize", text_qa_template=prompt_template
        )
        query_engine = RetrieverQueryEngine(
            retriever=retriever,
            response_synthesizer=synthesizer,
        )
        result = query_engine.query(question)
        context = [n.node.text for n in result.source_nodes]
        return AnswerEvent(question=question, answer=result.response, context=context)

    @step
    async def detect_hallucination(self, ctx: Context, ev: AnswerEvent) -> StopEvent:
        lettuce_client = LettuceClient("http://127.0.0.1:8000")
        hallucination_scores = lettuce_client.detect_token(
            contexts=ev.context, question=ev.question, answer=ev.answer
        )
        return StopEvent(
            result={
                "answer": ev.answer,
                "hallucination_scores": hallucination_scores.predictions,
            }
        )

In [7]:
url = "https://lilianweng.github.io/posts/2023-06-23-agent/"
index = create_index(url)

In [8]:
workflow = RAGWorkflow()
result = await workflow.run(index=index, question="What is Task Decomposition?")

In [9]:
display_output(result["hallucination_scores"])